# Milestone 5 — M5A: analytic particle geometry

**M5A**: analytic spheres, segment–sphere intersections, multi-particle events.

Plan: [`plans/milestone_05/05_particle_plan.md`](../../plans/milestone_05/05_particle_plan.md).  
Guidelines: [`docs/milestone_05_particle_guidelines.md`](../../docs/milestone_05_particle_guidelines.md).  
Next: `05b_ray_particle.ipynb`.

## Notes

- Particles are continuous-space spheres — **not** meshed.
- Intersection events are **construction inputs** for M5B clean/dirty pairs (not the source-correction ledger).
- Assessment is visual plus printed chord lengths for canonical cases.



In [ ]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[fem]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from gummybear.particles import (
    ParticleSet,
    ParticleSphere,
    intersect_segments_with_particles,
    segment_sphere_intersection,
)
from gummybear_validation.plotting import (
    plot_intersection_event,
    plot_segment,
    plot_sphere,
    set_axes_equal,
)


# 1. Definition of an analytical particle

The baseline M5 particle is an analytic sphere defined by center, radius,
absorption coefficient and scattering coefficient.


In [ ]:
particle = ParticleSphere(
    center=np.array([0.0, 0.0, 0.0]),
    radius=1.0,
    mu_abs=5.0,
    mu_scat=2.0,
)

particle


# 2. Single ray–sphere intersection

A single ray segment intersecting the sphere produces an entry point,
an exit point, and an inside-particle chord.


In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")

plot_sphere(ax, particle, color="tab:blue", label="particle")
start = np.array([-2.0, 0.0, 0.0])
end = np.array([2.0, 0.5, 0.0])

event = segment_sphere_intersection(start, end, particle.center, particle.radius)

plot_segment(
    ax,
    event.entry_point,
    event.exit_point,
    color="red",
    linewidth=4,
    label="inside particle",
)

set_axes_equal(ax)
ax.legend()
plt.show()


# 3. Canonical intersection cases

The implementation handles the standard geometric cases described in the M5A plan.


In [ ]:
cases = {
    "centerline": (np.array([-2, 0, 0]), np.array([2, 0, 0])),
    "tangent": (np.array([-2, 1, 0]), np.array([2, 1, 0])),
    "miss": (np.array([-2, 2, 0]), np.array([2, 2, 0])),
    "starts_inside": (np.array([0, 0, 0]), np.array([2, 0, 0])),
    "fully_inside": (np.array([-0.25, 0, 0]), np.array([0.25, 0, 0])),
}

for name, (start, end) in cases.items():
    event = segment_sphere_intersection(start, end, particle.center, particle.radius)
    print("=" * 60)
    print(name)
    if event is None:
        print("MISS")
    else:
        print(f"path_length={event.path_length_inside_particle:.6f}")


# 4. Multiple particles

Particles are grouped into a `ParticleSet`, which serves as the basic M5
container for particle realizations.


In [ ]:
particles = ParticleSet.from_particles([
    ParticleSphere(center=[-1.0, 0.0, 0.0], radius=0.3),
    ParticleSphere(center=[0.0, 0.0, 0.0], radius=0.5),
    ParticleSphere(center=[1.0, 0.0, 0.0], radius=0.3),
])

particles.to_manifest_block()


# 5. Multiple segments intersecting multiple particles

Intersection events are generated independently for each segment-particle
overlap and retain the corresponding segment and particle indices.


In [ ]:
starts = np.array([[-2.0, 0.0, 0.0], [0.0, -2.0, 0.0]])
ends = np.array([[2.0, 0.0, 0.0], [0.0, 2.0, 0.0]])

events = intersect_segments_with_particles(starts, ends, particles)

for e in events:
    print(
        f"segment={e.segment_index:2d} "
        f"particle={e.particle_index:2d} "
        f"entry={e.entry_t:.3f} "
        f"exit={e.exit_t:.3f}"
    )


# 6. Visualization of intersection events

Colored spheres represent particles; black lines represent transport
segments; highlighted chords indicate the portions of segments lying
inside particles.


In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")

for i, p in enumerate(particles):
    plot_sphere(ax, p, color=f"C{i}", label=f"particle {i}")

for i in range(len(starts)):
    plot_segment(ax, starts[i], ends[i], color="black", linewidth=2.0, label=f"segment {i}")

for i, event in enumerate(events):
    plot_intersection_event(
        ax,
        event,
        label_chord="inside-particle chord" if i == 0 else None,
    )

ax.set_title("M5A: Particle-Segment Intersection Geometry")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
set_axes_equal(ax)
ax.legend()
plt.show()


# Conclusion M5A

Milestone 5A establishes analytic particle geometry and segment-particle
intersection detection.

Analytic spheres can be grouped into `ParticleSet`s, intersection events
can be computed for individual or multiple particles, and the resulting
inside-particle chords are geometrically consistent with the underlying
ray segments.

These intersection events are the construction inputs for M5B.
